In [6]:
!pip -q install torch torchvision scikit-learn opencv-python-headless tqdm


In [7]:
import os
from pathlib import Path

!wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz
!tar -xzf imagenette2-320.tgz

DATA_DIR = Path("/content/imagenette2-320")
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR = DATA_DIR / "val"

print("Training directory:", TRAIN_DIR)
print("Validation directory:", VAL_DIR)
print("Training classes:", len(list(TRAIN_DIR.iterdir())))
print("Validation classes:", len(list(VAL_DIR.iterdir())))

Training directory: /content/imagenette2-320/train
Validation directory: /content/imagenette2-320/val
Training classes: 10
Validation classes: 10


In [8]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ImageNet normalization for the pretrained ConvNeXt model
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_ds = datasets.ImageFolder(VAL_DIR, transform=eval_transform)

# Confirm class mappings match
assert train_ds.class_to_idx == val_ds.class_to_idx

print("Classes:", train_ds.classes)
print("Training images:", len(train_ds))
print("Validation images:", len(val_ds))

assert len(val_ds) == 3925, (
    f"Expected 3925 validation images, found {len(val_ds)}"
)

train_loader = DataLoader(
    train_ds, batch_size=32, shuffle=True,
    num_workers=2, pin_memory=True
)

val_loader = DataLoader(
    val_ds, batch_size=32, shuffle=False,
    num_workers=2, pin_memory=True
)

Device: cuda
Classes: ['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']
Training images: 9469
Validation images: 3925


In [9]:
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights

weights = ConvNeXt_Tiny_Weights.DEFAULT
model = convnext_tiny(weights=weights)

# Change the final classifier from 1,000 ImageNet classes to 10
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, len(train_ds.classes))

model = model.to(device)

print(model.classifier)

Sequential(
  (0): LayerNorm2d((768,), eps=1e-06, elementwise_affine=True)
  (1): Flatten(start_dim=1, end_dim=-1)
  (2): Linear(in_features=768, out_features=10, bias=True)
)


In [5]:
import random
import numpy as np
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

EPOCHS = 5  # Increase after confirming the pipeline works
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(
        f"Epoch {epoch+1}: "
        f"loss={running_loss/total:.4f}, "
        f"train_acc={train_acc:.4f}, "
        f"val_acc={val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": train_ds.classes,
            "class_to_idx": train_ds.class_to_idx,
            "seed": SEED,
            "epochs": EPOCHS
        }, "/content/convnext_tiny_imagenette.pth")

print("Best validation accuracy:", best_val_acc)

Epoch 1/5:   0%|          | 0/296 [00:00<?, ?it/s]

Epoch 1: loss=0.3294, train_acc=0.9195, val_acc=0.9893


Epoch 2/5:   0%|          | 0/296 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 2: loss=0.1189, train_acc=0.9631, val_acc=0.9934


...

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20><function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20>
Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
if w.is_alive():    
if w.is_alive():  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive

      File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
assert self._parent_pid == os.getpid(), 'can only test a

Epoch 4: loss=0.0893, train_acc=0.9750, val_acc=0.9941


Epoch 5/5:   0%|          | 0/296 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Epoch 5: loss=0.0878, train_acc=0.9742, val_acc=0.9954
Best validation accuracy: 0.9954140127388535


In [10]:
import cv2
import io
from PIL import Image
from torch.utils.data import Dataset

def opencv_cartoon(pil_image):
    """A reproducible OpenCV cartoon effect."""
    rgb = np.array(pil_image.convert("RGB"))
    img = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    # Smooth colours while preserving larger edges
    smooth = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 7)
    edges = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY,
        9, 2
    )

    cartoon = cv2.bitwise_and(smooth, smooth, mask=edges)
    cartoon = cv2.cvtColor(cartoon, cv2.COLOR_BGR2RGB)
    return Image.fromarray(cartoon)


def jpeg_compress(pil_image, quality):
    """JPEG encode/decode at a specified quality factor."""
    buffer = io.BytesIO()
    pil_image.convert("RGB").save(
        buffer, format="JPEG", quality=quality
    )
    buffer.seek(0)
    with Image.open(buffer) as decoded:
        return decoded.convert("RGB").copy()


class VariantDataset(Dataset):
    def __init__(self, image_folder, condition="original"):
        self.samples = image_folder.samples
        self.transform = eval_transform
        self.condition = condition

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        with Image.open(path) as im:
            image = im.convert("RGB")

        if self.condition == "opencv":
            image = opencv_cartoon(image)
        elif self.condition.startswith("jpeg_"):
            quality = int(self.condition.split("_")[1])
            image = jpeg_compress(image, quality)

        return self.transform(image), label

In [11]:
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    classification_report
)
import pandas as pd

checkpoint = torch.load(
    "/content/convnext_tiny_imagenette.pth",
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

conditions = [
    "original",
    "opencv",
    "jpeg_95",
    "jpeg_75",
    "jpeg_50",
    "jpeg_25"sni
]

results = []
all_predictions = {}

for condition in conditions:
    ds = VariantDataset(val_ds, condition=condition)
    loader = DataLoader(
        ds, batch_size=32, shuffle=False, num_workers=2
    )

    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc=condition):
            images = images.to(device)
            predictions = model(images).argmax(dim=1).cpu().numpy()

            y_pred.extend(predictions.tolist())
            y_true.extend(labels.numpy().tolist())

    assert len(y_true) == 3925

    all_predictions[condition] = np.array(y_pred)

    results.append({
        "Condition": condition,
        "Images": len(y_true),
        "Top-1 Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred)
    })

    print(f"\n--- {condition} ---")
    print(classification_report(
        y_true, y_pred,
        labels=list(range(len(val_ds.classes))),
        target_names=val_ds.classes,
        zero_division=0
    ))

results_df = pd.DataFrame(results)
display(results_df.round(4))

results_df.to_csv(
    "/content/convnext_results.csv", index=False
)
print("Saved results to /content/convnext_results.csv")

original:   0%|          | 0/123 [00:00<?, ?it/s]


--- original ---
              precision    recall  f1-score   support

   n01440764       1.00      1.00      1.00       387
   n02102040       1.00      1.00      1.00       395
   n02979186       0.99      0.99      0.99       357
   n03000684       0.98      0.99      0.99       386
   n03028079       1.00      1.00      1.00       409
   n03394916       0.99      1.00      1.00       394
   n03417042       0.99      0.99      0.99       389
   n03425413       0.99      0.99      0.99       419
   n03445777       1.00      0.99      1.00       399
   n03888257       1.00      0.99      0.99       390

    accuracy                           1.00      3925
   macro avg       1.00      1.00      1.00      3925
weighted avg       1.00      1.00      1.00      3925



opencv:   0%|          | 0/123 [00:00<?, ?it/s]


--- opencv ---
              precision    recall  f1-score   support

   n01440764       1.00      0.40      0.57       387
   n02102040       0.99      0.87      0.92       395
   n02979186       0.95      0.75      0.84       357
   n03000684       0.38      0.97      0.54       386
   n03028079       1.00      0.90      0.95       409
   n03394916       0.96      0.85      0.90       394
   n03417042       0.99      0.54      0.70       389
   n03425413       0.80      0.88      0.84       419
   n03445777       0.92      0.94      0.93       399
   n03888257       0.97      0.88      0.92       390

    accuracy                           0.80      3925
   macro avg       0.90      0.80      0.81      3925
weighted avg       0.90      0.80      0.81      3925



jpeg_95:   0%|          | 0/123 [00:00<?, ?it/s]

...

jpeg_25:   0%|          | 0/123 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20>Exception ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7bff6c8e5b20>Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__

Traceback (most recent call last):
    self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__

      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    self._shutdown_workers()if w.is_alive():

  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
        if w.is_alive():assert self._parent_pid == os.getpid(), 'can only test a child process'Exception ignored in: 

<function _MultiProcessingDataLoaderIter.


--- jpeg_25 ---
              precision    recall  f1-score   support

   n01440764       1.00      1.00      1.00       387
   n02102040       1.00      0.99      1.00       395
   n02979186       0.99      0.99      0.99       357
   n03000684       0.97      0.98      0.98       386
   n03028079       1.00      0.99      1.00       409
   n03394916       0.99      1.00      1.00       394
   n03417042       0.99      0.99      0.99       389
   n03425413       0.98      0.99      0.98       419
   n03445777       1.00      0.99      1.00       399
   n03888257       0.99      1.00      0.99       390

    accuracy                           0.99      3925
   macro avg       0.99      0.99      0.99      3925
weighted avg       0.99      0.99      0.99      3925



,Condition,Images,Top-1 Accuracy,Macro-F1,Balanced Accuracy
0,original,3925,0.9954,0.9954,0.9954
1,opencv,3925,0.8008,0.8125,0.7986
2,jpeg_95,3925,0.9949,0.9949,0.9949
3,jpeg_75,3925,0.9952,0.9951,0.9951
4,jpeg_50,3925,0.9939,0.9939,0.9939
5,jpeg_25,3925,0.9921,0.9921,0.9921


Saved results to /content/convnext_results.csv


In [12]:
original_preds = all_predictions["original"]

flip_results = []

for condition, predictions in all_predictions.items():
    changed = predictions != original_preds

    flip_results.append({
        "Condition": condition,
        "Changed Predictions": int(changed.sum()),
        "Prediction Flip Rate": float(changed.mean())
    })

flip_df = pd.DataFrame(flip_results)
display(flip_df.round(4))

flip_df.to_csv(
    "/content/convnext_prediction_flips.csv", index=False
)

,Condition,Changed Predictions,Prediction Flip Rate
0,original,0,0.0000
1,opencv,774,0.1972
2,jpeg_95,2,0.0005
3,jpeg_75,1,0.0003
4,jpeg_50,8,0.0020
5,jpeg_25,19,0.0048
